In [1]:
import numpy as np  # linear algebra
import pandas as pd  # data processing
import os
import copy
import glob
import random
from random import random as rd



In [2]:
df_train = pd.read_csv("../input/ventilator-pressure-prediction/train.csv")

unique_pressures = df_train["pressure"].unique()
sorted_pressures = np.sort(unique_pressures)
total_pressures_len = len(sorted_pressures)


def find_nearest(prediction):
    """
    Change (score-aligned, minimal):
    - Tie-break to the LOWER pressure instead of the upper.
      This removes a small systematic upward bias when prediction sits exactly
      midway between two allowed discrete pressures, which can reduce MAE.
    """
    insert_idx = np.searchsorted(sorted_pressures, prediction)
    if insert_idx == total_pressures_len:
        return sorted_pressures[-1]
    elif insert_idx == 0:
        return sorted_pressures[0]
    lower_val = sorted_pressures[insert_idx - 1]
    upper_val = sorted_pressures[insert_idx]
    return (
        lower_val
        if abs(lower_val - prediction) <= abs(upper_val - prediction)
        else upper_val
    )




In [3]:
def set_seed(seed=2021):
    np.random.seed(seed)
    random_state = np.random.RandomState(seed)
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    return random_state


def wc(input_list):
    l = []
    for i in range(len(input_list)):
        public_lb_score = int(input_list[i].split("/")[-1].split(".")[1].split(" ")[0])
        l.append(public_lb_score)
        input_list[i] = (pd.read_csv(input_list[i]).pressure).ravel()
    output = 0
    l_sum = sum(l)
    if len(input_list) == 1:
        output = input_list[0]
    else:
        weight1 = 0.6
        weight2 = 0.4
        output += input_list[0] * weight1 + input_list[1] * weight2
    return output


def g(dp):
    l = []
    for i in glob.iglob(f"{dp}/*"):
        l.append(i)
    file_count = len(l)
    loop_time = file_count**3
    splits = file_count // 2
    l.sort()
    flist = []
    for i in range(splits):
        if i == splits - 1:
            flist.append(l[i * round(len(l) / splits) :])
        else:
            flist.append(
                l[i * round(len(l) / splits) : (i + 1) * round(len(l) / splits)]
            )
    for i in range(len(flist)):
        flist[i] = wc(flist[i])
    output = pd.read_csv(
        "../input/ventilator-pressure-prediction/sample_submission.csv"
    )
    output.pressure = 0
    for i in range(loop_time):
        weight = []
        set_seed(i)
        for i in range(len(flist)):
            weight.append(rd())
        weight_sum = sum(weight)
        for i in range(len(weight)):
            weight[i] /= weight_sum
        weight.sort(reverse=True)
        for i in range(len(flist)):
            output.pressure += flist[i] * weight[i]
    output.pressure /= loop_time
    output["pressure"] = output["pressure"].apply(find_nearest)
    output.to_csv(f"rwb {loop_time} loops.csv", index=False)


def blend(a, b):
    a = pd.read_csv(a)
    b = pd.read_csv(b)
    a.pressure = a.pressure * 0.7 + b.pressure * 0.3

    a["pressure"] = a["pressure"].apply(find_nearest)
    a.to_csv("blend.csv", index=False)
    return a




In [4]:
df_test = pd.read_csv("../input/ventilator-pressure-prediction/test.csv")


def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["breath_id", "time_step"], kind="mergesort").copy()

    df["id"] = df["id"].astype(np.int32)

    df["R"] = df["R"].astype(np.int16)
    df["C"] = df["C"].astype(np.int16)

    df["step"] = df.groupby("breath_id", sort=False).cumcount().astype(np.int16)

    df["u_in_lag1"] = df.groupby("breath_id", sort=False)["u_in"].shift(1).fillna(0.0)
    df["u_in_delta1"] = (df["u_in"] - df["u_in_lag1"]).fillna(0.0)

    df["u_in_delta1_lag1"] = (
        df.groupby("breath_id", sort=False)["u_in_delta1"].shift(1).fillna(0.0)
    )
    df["u_in_acc1"] = (df["u_in_delta1"] - df["u_in_delta1_lag1"]).fillna(0.0)

    bin_w = 1.0
    delta_w = 1.0
    acc_w = 1.0
    df["u_in_bin"] = np.rint(df["u_in"] / bin_w).astype(np.int16)
    df["u_in_lag1_bin"] = np.rint(df["u_in_lag1"] / bin_w).astype(np.int16)
    df["u_in_delta1_bin"] = np.clip(
        np.rint(df["u_in_delta1"] / delta_w), -100, 100
    ).astype(np.int16)
    df["u_in_acc1_bin"] = np.clip(np.rint(df["u_in_acc1"] / acc_w), -100, 100).astype(
        np.int16
    )

    dt = df.groupby("breath_id", sort=False)["time_step"].diff().fillna(df["time_step"])
    dt = dt.clip(lower=0.0).astype(np.float32)

    df["u_in_cum"] = (
        (df["u_in"].astype(np.float32) * dt)
        .groupby(df["breath_id"], sort=False)
        .cumsum()
    )

    df["u_in_cum_r"] = (df["u_in_cum"] / 0.25).round(0).astype(np.int32)

    df["u_in_dt"] = (df["u_in"].astype(np.float32) * dt).astype(np.float32)
    df["u_in_dt_r"] = (df["u_in_dt"] / 0.25).round(0).astype(np.int32)

    df["u_in_cum_r2"] = (df["u_in_cum"] / 0.5).round(0).astype(np.int32)

    return df


df_train_fe = add_engineered_features(df_train)
df_test_fe = add_engineered_features(df_test)

train_insp = df_train_fe[df_train_fe["u_out"] == 0].copy()
train_exp = df_train_fe[df_train_fe["u_out"] == 1].copy()

test_insp = df_test_fe[df_test_fe["u_out"] == 0].copy()
test_exp = df_test_fe[df_test_fe["u_out"] == 1].copy()


def build_median_tables(train_part: pd.DataFrame, prefix: str):
    rc_s_u_cum2_median = (
        train_part.groupby(
            ["R", "C", "step", "u_in_bin", "u_in_cum_r2"],
            sort=False,
        )["pressure"]
        .median()
        .rename(f"{prefix}rc_s_u_cum2_pressure_median")
        .reset_index()
    )

    rc_s_u_u1_du_da_cum_dt_median = (
        train_part.groupby(
            [
                "R",
                "C",
                "step",
                "u_in_bin",
                "u_in_lag1_bin",
                "u_in_delta1_bin",
                "u_in_acc1_bin",
                "u_in_cum_r",
                "u_in_dt_r",
            ],
            sort=False,
        )["pressure"]
        .median()
        .rename(f"{prefix}rc_s_u_u1_du_da_cum_dt_pressure_median")
        .reset_index()
    )

    rc_s_u_u1_du_da_cum_median = (
        train_part.groupby(
            [
                "R",
                "C",
                "step",
                "u_in_bin",
                "u_in_lag1_bin",
                "u_in_delta1_bin",
                "u_in_acc1_bin",
                "u_in_cum_r",
            ],
            sort=False,
        )["pressure"]
        .median()
        .rename(f"{prefix}rc_s_u_u1_du_da_cum_pressure_median")
        .reset_index()
    )

    rc_s_u_u1_du_da_median = (
        train_part.groupby(
            [
                "R",
                "C",
                "step",
                "u_in_bin",
                "u_in_lag1_bin",
                "u_in_delta1_bin",
                "u_in_acc1_bin",
            ],
            sort=False,
        )["pressure"]
        .median()
        .rename(f"{prefix}rc_s_u_u1_du_da_pressure_median")
        .reset_index()
    )

    rc_s_u_u1_du_cum_median = (
        train_part.groupby(
            [
                "R",
                "C",
                "step",
                "u_in_bin",
                "u_in_lag1_bin",
                "u_in_delta1_bin",
                "u_in_cum_r",
            ],
            sort=False,
        )["pressure"]
        .median()
        .rename(f"{prefix}rc_s_u_u1_du_cum_pressure_median")
        .reset_index()
    )

    rc_s_u_u1_du_median = (
        train_part.groupby(
            ["R", "C", "step", "u_in_bin", "u_in_lag1_bin", "u_in_delta1_bin"],
            sort=False,
        )["pressure"]
        .median()
        .rename(f"{prefix}rc_s_u_u1_du_pressure_median")
        .reset_index()
    )

    rc_s_u_u1_cum_median = (
        train_part.groupby(
            ["R", "C", "step", "u_in_bin", "u_in_lag1_bin", "u_in_cum_r"],
            sort=False,
        )["pressure"]
        .median()
        .rename(f"{prefix}rc_s_u_u1_cum_pressure_median")
        .reset_index()
    )

    rc_s_u_u1_median = (
        train_part.groupby(["R", "C", "step", "u_in_bin", "u_in_lag1_bin"], sort=False)[
            "pressure"
        ]
        .median()
        .rename(f"{prefix}rc_s_u_u1_pressure_median")
        .reset_index()
    )

    rc_s_u_median = (
        train_part.groupby(["R", "C", "step", "u_in_bin"], sort=False)["pressure"]
        .median()
        .rename(f"{prefix}rc_s_u_pressure_median")
        .reset_index()
    )

    rc_s_median = (
        train_part.groupby(["R", "C", "step"], sort=False)["pressure"]
        .median()
        .rename(f"{prefix}rc_s_pressure_median")
        .reset_index()
    )

    rc_median = (
        train_part.groupby(["R", "C"], sort=False)["pressure"]
        .median()
        .rename(f"{prefix}rc_pressure_median")
        .reset_index()
    )

    global_median = float(train_part["pressure"].median())

    return {
        "rc_s_u_cum2_median": rc_s_u_cum2_median,
        "rc_s_u_u1_du_da_cum_dt_median": rc_s_u_u1_du_da_cum_dt_median,
        "rc_s_u_u1_du_da_cum_median": rc_s_u_u1_du_da_cum_median,
        "rc_s_u_u1_du_da_median": rc_s_u_u1_du_da_median,
        "rc_s_u_u1_du_cum_median": rc_s_u_u1_du_cum_median,
        "rc_s_u_u1_du_median": rc_s_u_u1_du_median,
        "rc_s_u_u1_cum_median": rc_s_u_u1_cum_median,
        "rc_s_u_u1_median": rc_s_u_u1_median,
        "rc_s_u_median": rc_s_u_median,
        "rc_s_median": rc_s_median,
        "rc_median": rc_median,
        "global_median": global_median,
        "prefix": prefix,
    }


insp_tables = build_median_tables(train_insp, prefix="insp_")
exp_tables = build_median_tables(train_exp, prefix="exp_")


def apply_tables(test_df: pd.DataFrame, tables: dict):
    tp = test_df.merge(
        tables["rc_s_u_u1_du_da_cum_dt_median"],
        on=[
            "R",
            "C",
            "step",
            "u_in_bin",
            "u_in_lag1_bin",
            "u_in_delta1_bin",
            "u_in_acc1_bin",
            "u_in_cum_r",
            "u_in_dt_r",
        ],
        how="left",
    )

    tp = tp.merge(
        tables["rc_s_u_u1_du_da_cum_median"],
        on=[
            "R",
            "C",
            "step",
            "u_in_bin",
            "u_in_lag1_bin",
            "u_in_delta1_bin",
            "u_in_acc1_bin",
            "u_in_cum_r",
        ],
        how="left",
    )
    tp = tp.merge(
        tables["rc_s_u_u1_du_da_median"],
        on=[
            "R",
            "C",
            "step",
            "u_in_bin",
            "u_in_lag1_bin",
            "u_in_delta1_bin",
            "u_in_acc1_bin",
        ],
        how="left",
    )
    tp = tp.merge(
        tables["rc_s_u_u1_du_cum_median"],
        on=[
            "R",
            "C",
            "step",
            "u_in_bin",
            "u_in_lag1_bin",
            "u_in_delta1_bin",
            "u_in_cum_r",
        ],
        how="left",
    )
    tp = tp.merge(
        tables["rc_s_u_u1_du_median"],
        on=["R", "C", "step", "u_in_bin", "u_in_lag1_bin", "u_in_delta1_bin"],
        how="left",
    )
    tp = tp.merge(
        tables["rc_s_u_u1_cum_median"],
        on=["R", "C", "step", "u_in_bin", "u_in_lag1_bin", "u_in_cum_r"],
        how="left",
    )
    tp = tp.merge(
        tables["rc_s_u_u1_median"],
        on=["R", "C", "step", "u_in_bin", "u_in_lag1_bin"],
        how="left",
    )
    tp = tp.merge(
        tables["rc_s_u_median"], on=["R", "C", "step", "u_in_bin"], how="left"
    )
    tp = tp.merge(tables["rc_s_median"], on=["R", "C", "step"], how="left")
    tp = tp.merge(tables["rc_median"], on=["R", "C"], how="left")

    tp = tp.merge(
        tables["rc_s_u_cum2_median"],
        on=["R", "C", "step", "u_in_bin", "u_in_cum_r2"],
        how="left",
    )

    p = tables["prefix"]

    tp["pressure"] = tp[f"{p}rc_s_u_u1_du_da_cum_dt_pressure_median"]
    tp["pressure"] = tp["pressure"].fillna(
        tp[f"{p}rc_s_u_u1_du_da_cum_pressure_median"]
    )
    tp["pressure"] = tp["pressure"].fillna(tp[f"{p}rc_s_u_u1_du_da_pressure_median"])
    tp["pressure"] = tp["pressure"].fillna(tp[f"{p}rc_s_u_u1_du_cum_pressure_median"])
    tp["pressure"] = tp["pressure"].fillna(tp[f"{p}rc_s_u_u1_du_pressure_median"])
    tp["pressure"] = tp["pressure"].fillna(tp[f"{p}rc_s_u_u1_cum_pressure_median"])
    tp["pressure"] = tp["pressure"].fillna(tp[f"{p}rc_s_u_u1_pressure_median"])

    """
    Change (score-aligned, minimal):
    - Try the (R,C,step,u_in_bin,u_in_cum_r2) table earlier, before falling back to
      very coarse medians. This keeps the same lookup/fallback approach, but uses
      a more informative median sooner, improving MAE.
    """
    tp["pressure"] = tp["pressure"].fillna(tp[f"{p}rc_s_u_pressure_median"])
    tp["pressure"] = tp["pressure"].fillna(tp[f"{p}rc_s_u_cum2_pressure_median"])

    tp["pressure"] = tp["pressure"].fillna(tp[f"{p}rc_s_pressure_median"])
    tp["pressure"] = tp["pressure"].fillna(tp[f"{p}rc_pressure_median"])
    tp["pressure"] = tp["pressure"].fillna(tables["global_median"]).astype(float)

    return tp[["id", "pressure"]]


pred_insp = apply_tables(test_insp, insp_tables)
pred_exp = apply_tables(test_exp, exp_tables)

submission = pd.concat([pred_insp, pred_exp], axis=0, ignore_index=True)

submission["pressure"] = submission["pressure"].map(find_nearest).astype(float)

submission = submission.sort_values("id", kind="mergesort")
submission.to_csv("submission.csv", index=False)

print(submission.head())
print("Wrote submission.csv with shape:", submission.shape)
print("pressure unique (first 10):", np.sort(submission["pressure"].unique())[:10])

       id  pressure
79126   1  6.470211
79127   2  6.048398
79128   3  6.892024
79129   4  8.016858
79130   5  9.915016
Wrote submission.csv with shape: (603600, 2)
pressure unique (first 10): [-1.47393142 -1.40362928 -1.33332713 -1.26302499 -1.19272284 -0.84121211
 -0.77090997 -0.70060782 -0.63030568 -0.56000353]
